# Prediction under sensor loss (public data)

Hold out one quarter of participants. Train regularized logistic classifiers on intact measurements or on intact and masked copies. Test both models with all features and with the second sensor group replaced by the training mean. Compare AUROC under sensor loss. Use PhysioNet EEG motor imagery runs 4 and 8 for subjects 1–16. Extract log band power from eighteen electrodes in three bands and re-reference each side independently. The target is imagined left versus right fist. Withholding right-side sensors tests missing-view robustness on a behavioral task.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'missing-eeg-sensors-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'edfio==0.4.18']],check=True)

In [ ]:
(root / 'prepare.py').write_text('"""Cache the first 16 participants\' unilateral imagery recordings on the data drive."""\n\nimport os\nimport time\nimport urllib.request\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nDATA = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\nDATA = DATA / "eegmmidb"\nBASE = "https://physionet.org/files/eegmmidb/1.0.0"\n\n\ndef fetch(key):\n    subject, run = key\n    tag = f"S{subject:03d}"\n    name = f"{tag}R{run:02d}.edf"\n    target = DATA / name\n    if target.exists():\n        return target\n    for attempt in range(3):\n        try:\n            with urllib.request.urlopen(f"{BASE}/{tag}/{name}", timeout=45) as response:\n                data = response.read()\n                assert len(data) == int(\n                    response.headers.get("Content-Length", len(data))\n                )\n                assert data.startswith(b"0       ") and len(data) > 1000000\n            temp = target.with_suffix(".part")\n            temp.write_bytes(data)\n            temp.replace(target)\n            return target\n        except OSError:\n            if attempt == 2:\n                raise\n            time.sleep(2)\n\n\nif __name__ == "__main__":\n    DATA.mkdir(parents=True, exist_ok=True)\n    with ThreadPoolExecutor(4) as pool:\n        for path in pool.map(fetch, [(i, r) for i in range(1, 17) for r in (4, 8)]):\n            print(path.name, flush=True)\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nimport edfio\nfrom scipy.signal import butter, sosfiltfilt\n\n\ndef dataset(seed):\n    x = []\n    y = []\n    g = []\n    labels = "fc5 fc3 fc1 c5 c3 c1 cp5 cp3 cp1 fc6 fc4 fc2 c6 c4 c2 cp6 cp4 cp2".split()\n    for pid in range(1, 17):\n        for run_id in [4, 8]:\n            record = edfio.read_edf(DATA / "eegmmidb" / f"S{pid:03}R{run_id:02}.edf")\n            signals = {s.label.strip(".").lower(): s.data for s in record.signals}\n            a = np.stack([signals[k] for k in labels])\n            # Reference each hemisphere independently to avoid contamination across masked views.\n            for side in [a[:9], a[9:]]:\n                side -= side.mean(0)\n            bands = [\n                sosfiltfilt(butter(4, b, btype="bandpass", fs=160, output="sos"), a)\n                for b in [(7, 12), (12, 20), (20, 30)]\n            ]\n            for event in record.annotations:\n                if event.text not in ("T1", "T2"):\n                    continue\n                j = round((event.onset + 0.5) * 160)\n                x.append(\n                    np.stack(\n                        [np.log(b[:, j : j + 480].var(1) + 1e-12) for b in bands], 1\n                    ).ravel()\n                )\n                y.append(int(event.text == "T2"))\n                g.append(pid)\n    return np.stack(x), np.array(y), np.array(g)\n\n\nfrom sklearn.model_selection import GroupShuffleSplit\n\n\ndef run(seed):\n    x, y, g = dataset(seed)\n    tr, te = next(\n        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(x, y, g)\n    )\n    mean = x[tr].mean(0)\n    std = x[tr].std(0) + 1e-6\n    a = (x[tr] - mean) / std\n    b = (x[te] - mean) / std\n\n    def missing(z):\n        z = z.copy()\n        z[:, z.shape[1] // 2 :] = 0\n        return z\n\n    out = {}\n    for name in ["standard", "dropout"]:\n        f = a\n        target = y[tr]\n        if name == "dropout":\n            f = np.concatenate([a, missing(a)])\n            target = np.tile(target, 2)\n        model = LogisticRegression(C=0.08, max_iter=500, class_weight="balanced").fit(\n            f, target\n        )\n        for cond, z in [("intact", b), ("missing", missing(b))]:\n            out[name + "_" + cond] = scores(y[te], model.predict_proba(z)[:, 1])\n        save((mean, std, model, np.unique(g[te])), f"{name}-{seed}")\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())